# Diese sind meine Lösungen für die Aufgaben über Logik, Beweise und Problemlösen.

---

## Aufgabe 1:

**Gegeben:**


---


- 17 Kinder mit nichtnegativen ganzen Zahlen $x_1,\dots,x_{17}$.
- Jede beliebige 5‑Gruppe hat höchstens 25 Eier.
- Jede beliebige 3‑Gruppe hat mindestens 14 Eier.

**Gesucht:**


---


- Die mögliche gerade Gesamtzahl $S=\sum_{i=1}^{17} x_i$.

**Strategie:**

---

1. Mathematische Schranken für $S$ per Doppelzählung ableiten.
2. Nur die verbleibenden geraden Kandidaten für $S$ algorithmisch prüfen.
3. Für jeden Kandidaten ein ILP formulieren, das prüft, ob eine Verteilung $x_1,\dots,x_{17}$ existiert.

### Erklärung der Doppelzählung:

Summe über alle 5‑Gruppen: Jede 5‑Gruppe hat ≤ 25 Eier, es gibt $\binom{17}{5}$ solche Gruppen. Also gilt:

$
\sum_{\text{alle 5‑Gruppen}} \text{Eier} \le \binom{17}{5}\cdot 25.
$

---


Jedes Kind erscheint in genau $\binom{16}{4}$ 5‑Gruppen, daher ist die linke Seite gleich $\binom{16}{4}\,S$. Daraus folgt:


$
\binom{16}{4}\,S \le \binom{17}{5}\cdot 25.
$


---

Summe über alle 3‑Gruppen: Jede 3‑Gruppe hat ≥ 14 Eier, es gibt $\binom{17}{3}$ solche Gruppen. Also gilt:


$
\binom{16}{2}\,S \ge \binom{17}{3}\cdot 14.
$

---

Diese beiden Ungleichungen liefern numerische Schranken für $S$. Danach runden wir auf ganze, gerade Werte und prüfen nur diese Kandidaten weiter.

In [ ]:
import math
from itertools import combinations
import numpy as np
import pulp

def C(n, k):
  return math.comb(n, k)

upper = (C(17,5)*25) / C(16,4)
lower = (C(17,3)*14) / C(16,2)
print(upper, "|", lower)

cand = [s for s in range(math.ceil(lower), math.floor(upper)+1) if s%2==0]
print("Kandidaten S:", cand)

85.0 | 79.33333333333333
Kandidaten S: [80, 82, 84]


**Erläuterung:**

---

Diese Zelle rechnet die beiden Ungleichungen aus und erzeugt die Menge der ganzzahligen, geraden Kandidaten für $S$. Wir verwenden diese Kandidaten als Input für die algorithmische Prüfung. Das reduziert den Suchraum drastisch.

In [ ]:
idx = range(17)
triples = list(combinations(idx,3))
quints  = list(combinations(idx,5))

def check_all_3(x):
    arr = np.array(x)
    return all(arr[list(t)].sum() >= 14 for t in triples)

def check_all_5(x):
    arr = np.array(x)
    return all(arr[list(q)].sum() <= 25 for q in quints)

**Erläuterung:**  

---

- Wir erzeugen `triples` und `quints` einmal, um wiederholte Kombinationserzeugung zu vermeiden.  
- `check_all_3` und `check_all_5` sind einfache, deterministische Prüfungen, die eine gegebene Verteilung $x$ gegen die Gruppenbedingungen testen.  
- Diese Funktionen sind nützlich zur Validierung von Lösungen, die der ILP‑Solver liefert, und können auch in heuristischen Suchen verwendet werden.


In [ ]:
def exists_distribution_for_S(S, time_limit=10):
    prob = pulp.LpProblem('eggs', pulp.LpStatusOptimal)
    x = [pulp.LpVariable(f"x{i}", lowBound=0, cat='Integer') for i in range(17)]
    # triple constraints
    for t in triples:
        prob += sum(x[i] for i in t) >= 14
    # quint constraints
    for q in quints:
        prob += sum(x[i] for i in q) <= 25
    prob += sum(x) == S
    prob.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=time_limit))
    return pulp.LpStatus[prob.status], [v.value() for v in x] if pulp.LpStatus[prob.status]=='Optimal' else None

for S in cand:
    status, sol = exists_distribution_for_S(S)
    print(S, status)
    if sol:
        print(sol)
        break

80 Infeasible
82 Infeasible
84 Optimal
[5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 4.0, 5.0, 5.0, 5.0, 5.0]


**Erläuterung ILP:**  

---

- **Variablen:** $x_0,\dots,x_{16}$ ganzzahlig, ≥0.  
- **Nebenbedingungen:** Für jedes Tripel $\sum_{i\in T} x_i \ge 14$. Für jedes Quintett $\sum_{i\in Q} x_i \le 25$. Und die Summengleichung $\sum_i x_i = S$.  
- **Solver:** `pulp` mit CBC. `timeLimit` schützt vor zu langen Läufen.  
- **Warum ILP:** Die Nebenbedingungen sind linear und ganzzahlig; ILP ist eine saubere, deterministische Methode, um Existenz einer Verteilung zu prüfen, ohne alle Permutationen zu enumerieren.

### Lösung:

---

**Behauptung:** Es wurden **84** Ostereier bemalt.

**Beweis (schrittweise):**

1. **Notation.** Sei $x_1,\dots,x_{17}$ die von den 17 Kindern bemalten Eier und $S=\sum_{i=1}^{17}x_i$ die Gesamtzahl.

2. **Doppelzählung für 5‑Gruppen.** Die Summe der Eier über alle $\binom{17}{5}$ 5‑Kinder‑Gruppen ist höchstens $\binom{17}{5}\cdot 25$. Andererseits wird jedes Kind in genau $\binom{16}{4}$ solcher Gruppen gezählt, also gilt
$
\binom{16}{4}\,S \le \binom{17}{5}\cdot 25.
$

3. **Doppelzählung für 3‑Gruppen.** Die Summe der Eier über alle $\binom{17}{3}$ 3‑Kinder‑Gruppen ist mindestens $\binom{17}{3}\cdot 14$. Da jedes Kind in $\binom{16}{2}$ 3‑Gruppen vorkommt, folgt
$
\binom{16}{2}\,S \ge \binom{17}{3}\cdot 14.
$

4. **Einsetzen der Binomialkoeffizienten und numerische Schranken.** Mit $\binom{17}{5}=6188,\ \binom{16}{4}=1820,\ \binom{17}{3}=680,\ \binom{16}{2}=120$ erhält man
$
1820\,S \le 6188\cdot 25 =154700 \quad\Rightarrow\quad S\le 85,
$
$
120\,S \ge 680\cdot 14 =9520 \quad\Rightarrow\quad S\ge \frac{9520}{120}=79\frac{1}{3}.
$
Da $S$ ganzzahlig und zusätzlich gerade sein muss, bleiben nur die Kandidaten
$
S\in\{80,82,84\}.
$

5. **Ausschluss von 80 und 82.** Betrachte die kleinstmöglichen sinnvollen Einzelwerte: drei Kinder mit je 4 Eiern würden eine 3‑Gruppe mit Summe $12<14$ erzeugen. Um die 3‑Gruppenbedingung zu erfüllen, dürfen also nicht drei Kinder gleichzeitig den Wert 4 haben. Schreibe $S$ als Kombination von 4 und 5 (kleinste sinnvolle Werte): $S=4\cdot(17-k)+5k=68+k$, wobei $k$ die Anzahl der Kinder mit 5 Eiern ist.  
- Für $S=80$ wäre $k=12$, also gäbe es 5 Kinder mit 4 Eiern — daraus folgt zwangsläufig eine Dreiergruppe aus drei Kindern mit 4 Eiern, Widerspruch.  
- Für $S=82$ wäre $k=14$, also gäbe es 3 Kinder mit 4 Eiern — wiederum existiert eine Dreiergruppe mit Summe $12<14$, Widerspruch.

6. **Existenz für $S=84$.** Setze 16 Kinder mit je 5 Eiern und 1 Kind mit 4 Eiern. Dann ist
$
S=16\cdot 5 + 1\cdot 4 =84.
$
Jede 5‑Gruppe enthält höchstens fünf Kinder mit 5 Eiern, also hat jede 5‑Gruppe Summe $\le 25$. Jede 3‑Gruppe enthält mindestens zwei Kinder mit 5 und höchstens eine mit 4, also hat jede 3‑Gruppe Summe $\ge 5+5+4=14$. Damit sind beide Bedingungen erfüllt.

**Schlussfolgerung:** Die einzige mögliche gerade Gesamtzahl, die alle Bedingungen erfüllt, ist **$S=84$**.

## Aufgabe 2:

**Gegeben:**

---

- Alex, Bert, Chris machen je 4 Aussagen über ihre Fangzahlen $a, b, c \geq 0$.
- Von den 4 Aussagen jedes Jungen sind genau 2 wahr.

**Gesucht:**

---

- Die eindeutigen Werte von $a$, $b$, $c$.

**Strategie:**

---

1. Jede Aussage als Boolesche Funktion von $(a,b,c)$ kodieren.
2. Alle möglichen Wahrheitsmuster (je Person $\binom{4}{2}=6$ Muster, insgesamt $6^3=216$ Fälle), prüfen. Für jedes Muster die nötigen Gleichungen/Ungleichungen ableiten und passende $(a,b,c)$ suchen.
3. Prüfen, ob genau 2 von 4 Aussagen pro Person wahr sind.
4. Eindeutigkeit der Lösung überprüfen.

In [ ]:
from itertools import combinations, product

def statements_A(a,b,c):
    return [c == 2, b == c + 1, a + b == c + 8, a > b + c]

def statements_B(a,b,c):
    return [ (c > a and c > b), b == a + 3, b >= 1, a == c ]

def statements_C(a,b,c):
    return [ b == 0, c >= 3, c != a, a + b == 18 ]


pairs = list(combinations(range(4), 2))

solutions = set()

for A_true in pairs:
    for B_true in pairs:
        for C_true in pairs:
            for a in range(0, 31):
                for b in range(0, 31):
                    for c in range(0, 31):
                        A = statements_A(a,b,c)
                        B = statements_B(a,b,c)
                        C = statements_C(a,b,c)
                        if all((i in A_true) == A[i] for i in range(4)) and \
                           all((i in B_true) == B[i] for i in range(4)) and \
                           all((i in C_true) == C[i] for i in range(4)):
                            solutions.add((a,b,c))

solutions = sorted(solutions)
print("Anzahl Lösungen:", len(solutions))
for sol in solutions:
    print("Alex:", sol[0], "Bert:", sol[1], "Chris:", sol[2])


Anzahl Lösungen: 1
Alex: 7 Bert: 10 Chris: 9


**Erläuterung:**

---

- Wir kodieren jede der 12 Aussagen als Boolesche Funktion von $(a, b, c)$.
- Die verschachtelten Schleifen prüfen alle 216 Kombinationen von `pairs` (Index-Paare der wahren Aussagen).
- Die Bedingung `all((i in A_true) == A[i] for i in range(4))` stellt sicher, dass für Alex *genau* die zwei Aussagen aus `A_true` wahr sind und die anderen beiden falsch.
- Der Suchraum für die Fischzahlen $\{0, \dots, 30\}^3$ ist völlig ausreichend: Aus C4 folgt $a + b = 18$ (also $a, b \le 18$), und aus A3 folgt $a + b = c + 8$, was $c$ ebenfalls begrenzt.
- Wir prüfen alle Tripel deterministisch durch — ein ILP ist hier nicht nötig, da der Raum klein genug ist.

### Lösung:

---

**Behauptung:** Alex hat **7** Fische gefangen, Bert hat **10** Fische gefangen und Chris hat **9** Fische gefangen.

**Prozess (schrittweise):**

1. **Notation:** Seien $a, b$ und $c$ die Anzahlen der gefangenen Fische und $alex, bert$ und $chris$ Listen, mit jeweils vier boolesche Werte. Diese boolesche Werte geben an, ob die Aussagen $A_1...C_4$ erfüllt werden oder nicht.

2. **Brute-Force Testing:** Im Programm wird jede Kombination der Werte für $a,b,c$ in einem Suchfeld von $1...30$ generiert. Diese sind $31^3=29791$ Kombinationen.

3. **Überprüfen der Kombinationen:** Nun überprüfen wir, welche Kombinationen jeweils 2 als Werte von $alex, bert, chris$ haben. Diese werden von dem Programm ausgedruckt.

4. **Interpretation:** Wenn $alex, bert, chris$ alle jeweils den Wert 2 haben, bedeutet es, dass von jeder Person genau zwei Aussage erfüllt wurden. Alle Kombinationen, die diese Bedingung erfüllen, werden ausgedruckt. Wenn die Aufgabe eindeutig lösbar ist, wird nur eine Kombination ausgedruckt. Im diesen Fall wurde nur ein Wert ausgedruckt, weshalb man schlussfolgern kann, dass die Aufgabe eindeutig lösbar ist.

**Schlussfolgerung:** Die einzige mögliche Kombination für $a, b, c$, welches die in der Aufgabenstellung gestellten Voraussetzung erfüllt, ist $a=7, b=10, c=9$.

## Aufgabe 3:

**Gegeben:**


---


- Vier Mathematiker feiern Silvester. Ben sagt: Es ist genau $h$ Stunden, $m$ Minuten und $s$ Sekunden vor Mitternacht.
- $h$, $m$ und $s$ sind allesamt Primzahlen und erfüllen die Gleichung $3s = h + m$.
- Carlo ergänzt: Die Anzahl der **vollen Minuten** bis Mitternacht ist ebenfalls eine Primzahl.
- Dieter ergänzt: Die Anzahl der **Sekunden** bis Mitternacht ist ebenfalls eine Primzahl.

**Gesucht:**


---


- Der eindeutige Zeitpunkt $h{:}m{:}s$ vor Mitternacht, der alle vier Bedingungen gleichzeitig erfüllt.

**Strategie:**

---

1. Primzahlen in den gültigen Bereichen $h \in [2,23]$, $m,s \in [2,59]$ aufzählen.
2. Statt drei verschachtelter Schleifen: $(h,m)$ iterieren und $s = \frac{h+m}{3}$ analytisch ableiten — dadurch wird Bedingung B2 exakt und ohne Raten erzwungen.
3. Die verbleibenden Kandidaten gegen Bedingung B3 (volle Minuten $= 60h + m$ prim) filtern.
4. Abschließend gegen Bedingung B4 (Gesamtsekunden $= 3600h + 60m + s$ prim) filtern.
5. Eindeutigkeit der Lösung algorithmisch nachweisen.

### Erklärung der Zeitumrechnung und Bedingungen:

Ist es $h$ Stunden, $m$ Minuten und $s$ Sekunden vor Mitternacht, so beträgt die verbleibende Zeit in Sekunden:

$$
T = 3600h + 60m + s.
$$

---

Die Anzahl der **vollen Minuten** bis Mitternacht ist:

$$
M = \left\lfloor \frac{T}{60} \right\rfloor = 60h + m,
$$

da $s < 60$, also $\lfloor s/60 \rfloor = 0$.

---

Die vier Bedingungen lauten damit formal:

$$
\text{(B1)}\quad h,m,s \in \mathbb{P}, \quad h \in [2,23], \quad m,s \in [2,59],
$$
$$
\text{(B2)}\quad 3s = h + m,
$$
$$
\text{(B3)}\quad 60h + m \in \mathbb{P},
$$
$$
\text{(B4)}\quad 3600h + 60m + s \in \mathbb{P}.
$$

---

Der analytische Trick: Aus (B2) folgt $s = (h+m)/3$. Damit reduziert sich die Suche auf Paare $(h,m)$ mit $3 \mid (h+m)$ — $s$ ist dann kein freier Parameter mehr, sondern eindeutig bestimmt. Dies reduziert die Komplexität von $O(n^3)$ auf $O(n^2)$.

In [1]:
from sympy import isprime, primerange

# Bedingung B1: Primzahlen in den gültigen Bereichen
primes_h  = list(primerange(2, 24))  # Stunden: [2, 3, 5, 7, 11, 13, 17, 19, 23]
primes_ms = list(primerange(2, 60))  # Minuten/Sekunden: Primzahlen bis 59

# Bedingung B2: s analytisch aus (h+m)/3 ableiten
candidates = []
for h in primes_h:
    for m in primes_ms:
        if (h + m) % 3 == 0:           # s muss ganzzahlig sein
            s = (h + m) // 3
            if isprime(s) and s < 60:  # s muss prim und gültig sein
                candidates.append((h, m, s))

print(f"Kandidaten nach B1+B2: {len(candidates)}")
print(candidates)

Kandidaten nach B1+B2: 9
[(2, 7, 3), (2, 13, 5), (2, 19, 7), (2, 31, 11), (2, 37, 13), (3, 3, 2), (7, 2, 3), (13, 2, 5), (19, 2, 7)]


**Erläuterung:**

---

Statt einer dreifach verschachtelten Schleife iterieren wir nur über $(h,m)$-Paare und berechnen $s = (h+m)/3$ direkt. Die Bedingung `(h + m) % 3 == 0` stellt sicher, dass $s$ ganzzahlig ist; andernfalls scheidet das Paar sofort aus. Danach prüfen wir mit `isprime(s)`, ob $s$ prim ist. Dieser analytische Ansatz erzwingt (B2) ohne Raten und liefert 9 Kandidaten.

In [ ]:
# Bedingung B3 (Carlo): Anzahl der vollen Minuten bis Mitternacht ist prim
candidates = [(h, m, s) for h, m, s in candidates
              if isprime(h * 60 + m)]

print(f"Kandidaten nach B3: {len(candidates)}")
for h, m, s in candidates:
    print(f"  h={h}, m={m}, s={s}  →  M = {h*60+m}")

**Erläuterung:**

---

Carlosaussage liefert Bedingung (B3): Die Anzahl der vollen Minuten bis Mitternacht ist $M = 60h + m$. Da $s < 60$, gilt $\lfloor T/60 \rfloor = 60h + m$ exakt. Wir filtern alle Kandidaten, bei denen $60h+m$ keine Primzahl ist; von 9 bleiben 4 übrig. Auffällig: alle vier verbleibenden Tripel haben $h=2$.

In [ ]:
# Bedingung B4 (Dieter): Gesamtanzahl der Sekunden bis Mitternacht ist prim
candidates = [(h, m, s) for h, m, s in candidates
              if isprime(h * 3600 + m * 60 + s)]

print(f"Kandidaten nach B4: {len(candidates)}")
assert len(candidates) == 1, "Keine eindeutige Lösung!"

h, m, s = candidates[0]
T = h * 3600 + m * 60 + s
M = h * 60 + m

print(f"\nEindeutige Lösung: h={h}, m={m}, s={s}")
print(f"  Gleichung:           3·{s} = {3*s} = {h}+{m} = h+m  ✓")
print(f"  Volle Minuten:  M = {M}  →  prim: {isprime(M)}  ✓")
print(f"  Gesamtsekunden: T = {T}  →  prim: {isprime(T)}  ✓")
print(f"\n  Es ist 21:22:47 Uhr.")

**Erläuterung:**

---

Dieters Aussage liefert Bedingung (B4): Die Gesamtzahl der Sekunden $T = 3600h + 60m + s$ bis Mitternacht muss prim sein. Von den vier verbleibenden Kandidaten überlebt nur $(h,m,s) = (2,37,13)$ diesen Filter. Die `assert`-Anweisung macht die Eindeutigkeit algorithmisch nachprüfbar: ein zweites gültiges Tripel würde sofort einen Laufzeitfehler erzeugen. Mitternacht minus $T = 9433 \text{ s} = 2\text{ h}\,37\text{ m}\,13\text{ s}$ ergibt den Zeitpunkt **21:22:47 Uhr**.

### Lösung:

---

**Behauptung:** Der gesuchte Zeitpunkt ist **21:22:47 Uhr**, d.h. es ist $h=2$ Stunden, $m=37$ Minuten und $s=13$ Sekunden vor Mitternacht.

**Beweis (schrittweise):**

1. **Notation.** Seien $h, m, s \in \mathbb{P}$ die Stunden, Minuten und Sekunden vor Mitternacht, $T = 3600h + 60m + s$ die Gesamtzeit in Sekunden und $M = 60h + m$ die Anzahl der vollen Minuten.

2. **Zulässige Primzahlen (B1).** Da $h$ eine Stundenzahl $< 24$ ist, kommen nur $h \in \{2,3,5,7,11,13,17,19,23\}$ in Frage. Analog für $m$ und $s$ die Primzahlen bis 59.

3. **Analytische Ableitung von $s$ (B2).** Aus $3s = h+m$ folgt $s = (h+m)/3$. Damit $s$ ganzzahlig ist, muss $3 \mid (h+m)$ gelten. Für jedes solche Paar $(h,m)$ ist $s$ eindeutig bestimmt; es wird zusätzlich auf Primalität und Gültigkeit ($s < 60$) geprüft. Das liefert 9 Kandidaten.

4. **Bedingung von Carlo (B3).** Die Anzahl der vollen Minuten $M = 60h + m$ muss prim sein. Da $s < 60$, gilt $\lfloor T/60 \rfloor = 60h + m$ exakt. Dieser Filter reduziert die Kandidaten auf 4 Tripel, alle mit $h = 2$:
$$
(2,7,3),\quad (2,19,7),\quad (2,31,11),\quad (2,37,13).
$$

5. **Bedingung von Dieter (B4).** Die Gesamtsekundenzahl $T = 3600h + 60m + s$ muss prim sein. Wir prüfen die vier Kandidaten:
$$
T_{(2,7,3)}   = 7623 = 3 \cdot 2541 \quad \text{(nicht prim)},
$$
$$
T_{(2,19,7)}  = 8347 = 17 \cdot 491 \quad \text{(nicht prim)},
$$
$$
T_{(2,31,11)} = 9071 = 47 \cdot 193 \quad \text{(nicht prim)},
$$
$$
T_{(2,37,13)} = 9433 \qquad\qquad\quad\ \text{(prim)}.
$$
Es verbleibt genau ein Tripel: $(h,m,s) = (2,37,13)$.

6. **Verifikation.**
   - $h = 2$, $m = 37$, $s = 13$ sind allesamt prim. ✓
   - $3s = 3 \cdot 13 = 39 = 2 + 37 = h + m$. ✓
   - Volle Minuten: $M = 60 \cdot 2 + 37 = 157$ ist prim. ✓
   - Gesamtsekunden: $T = 7200 + 2220 + 13 = 9433$ ist prim. ✓

**Schlussfolgerung:** Die einzige mögliche Uhrzeit, die alle vier Bedingungen erfüllt, ist $\mathbf{21:22:47}$ **Uhr**.